In [6]:
# benchmark rechenzeiten der beiden proximity_max_freq funktionen

import numpy as np
import time
import matplotlib.pyplot as plt

# --- Deine Funktionen ---
def proximity_max_freq(
    x,
    sharpness_base=10.0,
    sharpness_growth=2.0,
    decay=0.1,
    max_freq=4,
    weight_exponent=1.0,
    freq_sharpness_exp=1.0,
    norm_x=1.0,
    output_max=1.0
):
    x = np.array(x)
    freqs = np.arange(1, max_freq + 1)
    all_freq_values = []

    for freq in freqs:
        weight = 1.0 / (freq ** weight_exponent)
        freq_sharpness_base = sharpness_base * (freq ** freq_sharpness_exp)
        freq_sharpness_growth = sharpness_growth * (freq ** freq_sharpness_exp)

        sharpness = freq_sharpness_base * (x ** freq_sharpness_growth)
        cos_term = ((1 + np.cos(2 * np.pi * freq * x)) / 2) ** sharpness

        peak_sharpness = freq_sharpness_base * (norm_x ** freq_sharpness_growth)
        peak_value = ((1 + np.cos(2 * np.pi * freq * norm_x)) / 2) ** peak_sharpness
        peak_value = np.where(peak_value == 0, 1e-12, peak_value)

        denom = np.where(x == 0, 1e-12, x ** decay)
        val = weight * (cos_term / peak_value) / denom
        all_freq_values.append(val)

    all_freq_values = np.array(all_freq_values)
    proximity_max = np.max(all_freq_values, axis=0)

    max_at_norm_x = np.max([
        (1.0 / (freq ** weight_exponent)) * 1.0 / (norm_x ** decay)
        for freq in freqs
    ])
    proximity_max = proximity_max / max_at_norm_x * output_max
    return proximity_max

def proximity_max_freq_gaussian(
    x,
    sharpness_base=10.0,
    sharpness_growth=1.0,
    decay=0.1,
    max_freq=4,
    x_max=None,
    weight_exponent=1.0,
    freq_sharpness_exp=1.0,
    output_max=1.0,
    threshold=0.01,
):
    x = np.array(x, dtype=float)
    if x_max is None:
        x_max = int(np.ceil(np.max(x)))
    N = len(x)
    result = np.zeros(N)
    best_frac = [(0, 0)] * N

    for f in range(1, max_freq + 1):
        k_vals = np.arange(f, x_max * f + 1)
        mu_all = k_vals / f
        sharpness_f = sharpness_base * (f ** freq_sharpness_exp)
        sigma_all = 1.0 / (sharpness_f * (k_vals ** sharpness_growth) + 1e-9)
        weight_f = 1.0 / (f ** weight_exponent)
        mu_decay_all = 1.0 / (mu_all ** decay)

        diff = x[:, None] - mu_all[None, :]
        g = weight_f * mu_decay_all[None, :] * np.exp(-0.5 * (diff / sigma_all[None, :])**2)
        best_idx = np.argmax(g, axis=1)
        best_val = g[np.arange(N), best_idx]
        update_idx = best_val > result
        result[update_idx] = best_val[update_idx]
        for idx in np.where(update_idx)[0]:
            best_frac[idx] = (int(k_vals[best_idx[idx]]), f)

    if result.max() > 0:
        result /= result.max()
    result *= output_max
    result[result < threshold] = threshold
    return result, best_frac

# --- Benchmark-Skript ---
def benchmark(n_points=10000, repeats=5, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.random(n_points) * 5  # Zufallswerte zwischen 0 und 5

    times = {}
    for name, func in [
        ("cosine", proximity_max_freq),
        ("gaussian", proximity_max_freq_gaussian),
    ]:
        elapsed = []
        for _ in range(repeats):
            start = time.perf_counter()
            func(x)
            end = time.perf_counter()
            elapsed.append(end - start)
        times[name] = (np.mean(elapsed), np.std(elapsed))

    print("Benchmark Ergebnisse (Sekunden):")
    for name, (mean_t, std_t) in times.items():
        print(f"{name:10s}: {mean_t:.6f} ± {std_t:.6f}")

    # Optional: Verhältnis Cos vs Gauss
    speedup = times["gaussian"][0] / times["cosine"][0]
    print(f"\nSpeedup (gaussian / cosine): {speedup:.2f}x")

    return times

if __name__ == "__main__":
    benchmark(n_points=100000, repeats=10)

Benchmark Ergebnisse (Sekunden):
cosine    : 0.070345 ± 0.026415
gaussian  : 0.719603 ± 0.400845

Speedup (gaussian / cosine): 10.23x


In [4]:
# benchmark rechenzeiten der beiden proximity_max_freq funktionen

import numpy as np
import time
import matplotlib.pyplot as plt

# --- Deine Funktionen ---
def proximity_max_freq(
    x,
    sharpness_base=10.0,
    sharpness_growth=1.0,
    decay=0.1,
    max_freq=4,
    x_max=None,
    weight_exponent=1.0,
    freq_sharpness_exp=1.0,
    output_max=1.0,
    threshold=0.01,
):
    x = np.array(x, dtype=float)
    if x_max is None:
        x_max = int(np.ceil(np.max(x)))
    N = len(x)
    result = np.zeros(N)
    best_frac = [(0, 0)] * N

    for f in range(1, max_freq + 1):
        k_vals = np.arange(f, x_max * f + 1)
        mu_all = k_vals / f
        sharpness_f = sharpness_base * (f ** freq_sharpness_exp)
        sigma_all = 1.0 / (sharpness_f * (k_vals ** sharpness_growth) + 1e-9)
        weight_f = 1.0 / (f ** weight_exponent)
        mu_decay_all = 1.0 / (mu_all ** decay)

        diff = x[:, None] - mu_all[None, :]
        g = weight_f * mu_decay_all[None, :] * np.exp(-0.5 * (diff / sigma_all[None, :])**2)
        best_idx = np.argmax(g, axis=1)
        best_val = g[np.arange(N), best_idx]
        update_idx = best_val > result
        result[update_idx] = best_val[update_idx]
        for idx in np.where(update_idx)[0]:
            best_frac[idx] = (int(k_vals[best_idx[idx]]), f)

    if result.max() > 0:
        result /= result.max()
    result *= output_max
    result[result < threshold] = threshold
    return result, best_frac

def proximity_max_freq_gaussian(
    x,
    sharpness_base=10.0,
    sharpness_growth=1.0,
    decay=0.1,
    max_freq=4,
    weight_exponent=1.0,
    freq_sharpness_exp=1.0,
    neighbor_peaks=2,
    x_min=0.5,
    x_max=5.0,
    output_max=1.0
):
    """
    Gaussian-based proximity function with true neighbor-peak optimization.
    - Precomputes all relevant (k/f) peaks once
    - For each x, only evaluates the closest `neighbor_peaks`
    Complexity: O(N * neighbor_peaks)
    """
    x = np.array(x, dtype=float)

    # --- 1) Precompute all possible peaks (mu = k/f) in range ---
    peak_list = []
    for f in range(1, max_freq + 1):
        k_max = int(np.ceil(x_max * f))
        for k in range(1, k_max + 1):
            mu = k / f
            if mu < x_min or mu > x_max:
                continue
            weight = 1.0 / (f ** weight_exponent)
            sharpness = sharpness_base * (f ** freq_sharpness_exp) * (k ** sharpness_growth)
            sigma = 1.0 / (sharpness + 1e-9)
            mu_decay = 1.0 / (mu ** decay)
            peak_list.append((mu, sigma, weight * mu_decay))

    peak_list = np.array(peak_list, dtype=float)  # (N, 3): (mu, sigma, amplitude)
    mus, sigmas, amps = peak_list[:, 0], peak_list[:, 1], peak_list[:, 2]

    # --- 2) For each x, evaluate only closest peaks ---
    result = np.zeros_like(x, dtype=float)
    for i, xi in enumerate(x):
        # Find nearest peaks
        idx = np.argpartition(np.abs(mus - xi), neighbor_peaks)[:neighbor_peaks]
        g = np.exp(-0.5 * ((xi - mus[idx]) / sigmas[idx]) ** 2)
        result[i] = np.max(amps[idx] * g)

    # --- 3) Normalize ---
    result /= np.max(result) + 1e-9
    return result * output_max

# --- Benchmark-Skript ---
def benchmark(n_points=10000, repeats=5, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.random(n_points) * 5  # Zufallswerte zwischen 0 und 5

    times = {}
    for name, func in [
        ("cosine", proximity_max_freq),
        ("gaussian", proximity_max_freq_gaussian),
    ]:
        elapsed = []
        for _ in range(repeats):
            start = time.perf_counter()
            func(x)
            end = time.perf_counter()
            elapsed.append(end - start)
        times[name] = (np.mean(elapsed), np.std(elapsed))

    print("Benchmark Ergebnisse (Sekunden):")
    for name, (mean_t, std_t) in times.items():
        print(f"{name:10s}: {mean_t:.6f} ± {std_t:.6f}")

    # Optional: Verhältnis Cos vs Gauss
    speedup = times["gaussian"][0] / times["cosine"][0]
    print(f"\nSpeedup (gaussian / cosine): {speedup:.2f}x")

    return times

if __name__ == "__main__":
    benchmark(n_points=100000, repeats=10)

Benchmark Ergebnisse (Sekunden):
cosine    : 0.578286 ± 0.184589
gaussian  : 3.033282 ± 0.466523

Speedup (gaussian / cosine): 5.25x


In [7]:
# testtttt

# Bayesian Optimization mit globalen Nullmodellen aus Ordner mit counter
# Sucht nach der besten Parameterkombination für die Proximity-Funktion
# unter Verwendung von Bayesian Optimization.

# ================================
# Imports
# ================================
import os
import numpy as np
import pandas as pd
from skopt import gp_minimize
from skopt.space import Real, Integer
import matplotlib.pyplot as plt
import seaborn as sns
import time
from scipy.spatial.distance import cdist

# ================================
# Proximity-Funktion (Cosine)
# ================================

def proximity_cosine(X, sharpness_base=10.0, sharpness_growth=2.0, decay=1.0,
                     max_freq=5.0, weight_exponent=1.0, freq_sharpness_exp=2.0,
                     norm_x=1.0, output_max=1.0):
    """
    Cosine-basierte Proximity-Funktion.
    Gibt immer eine 2D-Matrix zurück (symmetrisch).
    """
    # Ensure input is 2D
    X = np.atleast_2d(X)

    # Abstandsmatrix (cosine basiert auf Winkeln -> aber hier euclidean als Bsp)
    distances = cdist(X, X, metric="euclidean")

    # Verhindere Overflows durch log-stabilisiertes Wachstum
    x = np.clip(distances / norm_x, 0, 50)
    sharpness = sharpness_base * np.exp(sharpness_growth * np.log1p(x))

    # Cosine-ähnliche Transformation
    sim = np.cos(np.minimum(x, max_freq) * np.pi / max_freq) ** weight_exponent

    # Decay einbauen
    sim *= np.exp(-decay * x)

    # Hochfrequente Schärfung
    sim **= freq_sharpness_exp

    # Skaliere auf [0, output_max]
    sim = np.clip(sim, 0, output_max)

    return sim


def proximity_gaussian(X, sharpness_base=10.0, sharpness_growth=2.0, decay=1.0,
                       max_freq=5.0, weight_exponent=1.0, freq_sharpness_exp=2.0,
                       norm_x=1.0, output_max=1.0):
    """
    Gaussian-basierte Proximity-Funktion.
    Gibt immer eine 2D-Matrix zurück (symmetrisch).
    """
    # Ensure input is 2D
    X = np.atleast_2d(X)

    # Abstandsmatrix (euclidean)
    distances = cdist(X, X, metric="euclidean")

    # Verhindere Overflows
    x = np.clip(distances / norm_x, 0, 50)
    sharpness = sharpness_base * np.exp(sharpness_growth * np.log1p(x))

    # Gaussian-Kern
    sim = np.exp(-(x ** 2) / (2.0 * sharpness ** 2))

    # Decay einbauen
    sim *= np.exp(-decay * x)

    # Exponentielle Gewichtung
    sim **= weight_exponent

    # Hochfrequente Schärfung
    sim **= freq_sharpness_exp

    # Skaliere auf [0, output_max]
    sim = np.clip(sim, 0, output_max)

    return sim

# ================================
# Folder-Analyse
# ================================
_already_printed_global_ioi = False

def analyze_folder_global_null(folder_path, params, num_sequences=500, seed=42, method="cosine"):
    """
    method = "cosine" oder "gaussian"
    """
    global _already_printed_global_ioi
    rng = np.random.default_rng(seed)
    results_list = []

    # Auswahl der Funktion
    if method == "cosine":
        prox_func_selected = proximity_cosine
    elif method == "gaussian":
        prox_func_selected = proximity_gaussian
    else:
        raise ValueError(f"Unknown method: {method}")

    # -----------------------------
    # 1) Alle IOIs laden und globale Parameter bestimmen
    # -----------------------------
    all_iois = []
    seqs = []

    for fname in os.listdir(folder_path):
        if not fname.endswith(".csv"):
            continue

        df = pd.read_csv(os.path.join(folder_path, fname))
        if "IOI" not in df.columns:
            continue

        iois = df["IOI"].dropna().values
        if len(iois) < 3:
            continue

        all_iois.extend(iois)
        seqs.append((fname, iois))

    all_iois = np.array(all_iois)
    global_median_ioi = np.median(all_iois)
    global_min = np.percentile(all_iois, 5)
    global_max = np.percentile(all_iois, 95)

    # nur beim allerersten Funktionsaufruf ausgeben
    if not _already_printed_global_ioi:
        print(f"Global median IOI = {global_median_ioi:.3f}")
        print(f"Uniform bounds = [{global_min:.3f}, {global_max:.3f}]")
        _already_printed_global_ioi = True
    
    # -----------------------------
    # 2) Nullmodelle vorbereiten (pro Sequenz-Länge!)
    # -----------------------------
    null_cache_expon = {}
    null_cache_uniform = {}
    null_cache_empirical = {}

    for fname, iois in seqs:
        n = len(iois)

        # Exponential Null
        if n not in null_cache_expon:
            medians_expon = []
            for _ in range(num_sequences):
                sim_iois = rng.exponential(scale=global_median_ioi, size=n)
                ratios = np.array([[max(sim_iois[i], sim_iois[j]) / min(sim_iois[i], sim_iois[j])
                                    for j in range(n)] for i in range(n)])
                prox_mat = prox_func_selected(ratios, **params)
                triu_idx = np.triu_indices_from(prox_mat, k=1)
                medians_expon.append(np.median(prox_mat[triu_idx]))
            null_cache_expon[n] = np.array(medians_expon)

        # Uniform Null
        if n not in null_cache_uniform:
            medians_uniform = []
            for _ in range(num_sequences):
                sim_iois = rng.uniform(global_min, global_max, size=n)
                ratios = np.array([[max(sim_iois[i], sim_iois[j]) / min(sim_iois[i], sim_iois[j])
                                    for j in range(n)] for i in range(n)])
                prox_mat = prox_func_selected(ratios, **params)
                triu_idx = np.triu_indices_from(prox_mat, k=1)
                medians_uniform.append(np.median(prox_mat[triu_idx]))
            null_cache_uniform[n] = np.array(medians_uniform)

        # Empirical Null (zieht zufällig IOIs aus ALLEN IOIs des Ordners)
        if n not in null_cache_empirical:
            medians_empirical = []
            for _ in range(num_sequences):
                sim_iois = rng.choice(all_iois, size=n, replace=True)
                ratios = np.array([[max(sim_iois[i], sim_iois[j]) / min(sim_iois[i], sim_iois[j])
                                    for j in range(n)] for i in range(n)])
                prox_mat = prox_func_selected(ratios, **params)
                triu_idx = np.triu_indices_from(prox_mat, k=1)
                medians_empirical.append(np.median(prox_mat[triu_idx]))
            null_cache_empirical[n] = np.array(medians_empirical)

    # -----------------------------
    # 3) Analyse jeder Sequenz mit den globalen Nullmodellen
    # -----------------------------
    for fname, iois in seqs:
        n = len(iois)
        ratios_real = np.array([[max(iois[i], iois[j]) / min(iois[i], iois[j])
                                 for j in range(n)] for i in range(n)])
        prox_mat_real = prox_func_selected(ratios_real, **params)
        triu_idx = np.triu_indices_from(prox_mat_real, k=1)
        proximity_real = prox_mat_real[triu_idx]
        median_real = np.median(proximity_real)

        # Exponential
        medians_expon = null_cache_expon[n]
        mean_expon = np.mean(medians_expon)
        std_expon = np.std(medians_expon, ddof=1)
        zscore_expon = (median_real - mean_expon) / std_expon if std_expon > 0 else np.nan
        pval_expon = (np.sum(medians_expon >= median_real) + 1) / (len(medians_expon) + 1)

        # Uniform
        medians_uniform = null_cache_uniform[n]
        mean_uniform = np.mean(medians_uniform)
        std_uniform = np.std(medians_uniform, ddof=1)
        zscore_uniform = (median_real - mean_uniform) / std_uniform if std_uniform > 0 else np.nan
        pval_uniform = (np.sum(medians_uniform >= median_real) + 1) / (len(medians_uniform) + 1)

        # Empirical
        medians_empirical = null_cache_empirical[n]
        mean_empirical = np.mean(medians_empirical)
        std_empirical = np.std(medians_empirical, ddof=1)
        zscore_empirical = (median_real - mean_empirical) / std_empirical if std_empirical > 0 else np.nan
        pval_empirical = (np.sum(medians_empirical >= median_real) + 1) / (len(medians_empirical) + 1)

        results_list.append({
            "filename": fname,
            "real_median": median_real,
            "expon_mean": mean_expon,
            "expon_std": std_expon,
            "uniform_mean": mean_uniform,
            "uniform_std": std_uniform,
            "empirical_mean": mean_empirical,   
            "empirical_std": std_empirical,     
            "diff_real_expon": median_real - mean_expon,
            "diff_real_uniform": median_real - mean_uniform,
            "diff_real_empirical": median_real - mean_empirical,   
            "zscore_expon": zscore_expon,
            "pval_expon": pval_expon,
            "zscore_uniform": zscore_uniform,
            "pval_uniform": pval_uniform,
            "zscore_empirical": zscore_empirical,   
            "pval_empirical": pval_empirical        
        })

    return pd.DataFrame(results_list)

# ================================
# Bayesian Optimization
# ================================

def bayesian_search_global_null(
    folder_path,
    n_calls=100,
    num_sequences=500,
    seed=42,
    objective="zscore",   # one of: "diff", "zscore", "pval"
    null_model="expon",   # name used in analyze_folder columns for now: "expon", "uniform", "empirical"
    method="cosine",      # <-- hinzugefügt
    param_space_def=None   # optional: override default param space dictionary
):
    """
    Bayesian search over the proximity-function parameter space.
    objective: "diff" (max), "zscore" (max), "pval" (min)
    null_model: string used to select columns returned by analyze_folder
    method: "cosine" oder "gaussian"
    """

    rng = np.random.default_rng(seed)
    default_params = {"norm_x": 1.0, "output_max": 1.0}

    # default parameter-space
    if param_space_def is None:
        param_space_def = {
            "sharpness_base": (1.0, 50.0),
            "sharpness_growth": (0.0, 5.0),
            "decay": (0.0, 4.0),
            "max_freq": (1, 6),  # integer
            "weight_exponent": (0.0, 4.0),
            "freq_sharpness_exp": (0.0, 4.0)
        }

    # build skopt space
    sk_space = [
        Real(param_space_def["sharpness_base"][0], param_space_def["sharpness_base"][1], name="sharpness_base"),
        Real(param_space_def["sharpness_growth"][0], param_space_def["sharpness_growth"][1], name="sharpness_growth"),
        Real(param_space_def["decay"][0], param_space_def["decay"][1], name="decay"),
        Integer(param_space_def["max_freq"][0], param_space_def["max_freq"][1], name="max_freq"),
        Real(param_space_def["weight_exponent"][0], param_space_def["weight_exponent"][1], name="weight_exponent"),
        Real(param_space_def["freq_sharpness_exp"][0], param_space_def["freq_sharpness_exp"][1], name="freq_sharpness_exp"),
    ]

    # will store intermediate results
    eval_records = []
    call_counter = {"n": 0}
    timing_info = {"start": None}

    # objective function for gp_minimize
    def objective_func(x):
        call_counter["n"] += 1

        # Startzeit merken
        if timing_info["start"] is None:
            timing_info["start"] = time.time()

        # vergangene Zeit berechnen
        elapsed = time.time() - timing_info["start"]
        avg_time = elapsed / call_counter["n"]
        remaining = avg_time * (n_calls - call_counter["n"])
        eta_min = int(remaining // 60)
        eta_sec = int(remaining % 60)

        print(f"[Bayesian Optimization] Iteration {call_counter['n']} / {n_calls} "
              f"(ETA: {eta_min}m {eta_sec}s)")

        # map x to param dict
        names = ["sharpness_base", "sharpness_growth", "decay", "max_freq",
                 "weight_exponent", "freq_sharpness_exp"]
        params = default_params.copy()
        params.update({n: v for n, v in zip(names, x)})
        params["max_freq"] = int(round(params["max_freq"]))

        # run analysis
        results_df = analyze_folder_global_null(folder_path, params,
                                                num_sequences=num_sequences, seed=seed,
                                                method=method)

        if results_df.empty:
            return 1e6

        col_map = {
            "diff": f"diff_real_{null_model}",
            "zscore": f"zscore_{null_model}",
            "pval": f"pval_{null_model}"
        }
        col = col_map[objective]
        if col not in results_df.columns:
            return 1e5

        vals = results_df[col].dropna().values
        if len(vals) == 0:
            return 1e5

        if objective in ("diff", "zscore"):
            agg = np.median(vals)
            score = -float(agg)
        else:  # pval
            agg = np.median(vals)
            score = float(agg)

        eval_records.append({**params, f"agg_{objective}_{null_model}": float(agg)})

        return score

    # run Bayesian optimization
    res = gp_minimize(objective_func, sk_space, n_calls=n_calls, random_state=seed)

    # collect evaluated params
    summary_eval_df = pd.DataFrame(eval_records)

    # determine best set
    agg_col_name = f"agg_{objective}_{null_model}"
    if agg_col_name in summary_eval_df.columns and not summary_eval_df.empty:
        if objective in ("diff", "zscore"):
            best_idx = summary_eval_df[agg_col_name].idxmax()
        else:  # pval
            best_idx = summary_eval_df[agg_col_name].idxmin()
        best_row = summary_eval_df.loc[best_idx]
    else:
        best_row = None

    return summary_eval_df, best_row, res

# ================================
# Optimierung + Sammeln der Sequenz-Werte
# ================================

def optimize_and_collect(folders, n_calls=30, num_sequences=50, seed=42, objective="pval"):
    all_seq_results = []
    methods = ["cosine", "gaussian"]

    for folder in folders:
        for method in methods:
            for null_model in ["expon", "uniform", "empirical"]:
                print("==============================")
                print(f"Starte Optimierung: Folder={folder}, Methode={method}, Nullmodell={null_model}, Objective={objective}")

                start_time = time.time()
                summary_eval_df, best_row, res = bayesian_search_global_null(
                    folder_path=folder,
                    n_calls=n_calls,
                    num_sequences=num_sequences,
                    seed=seed,
                    objective=objective,
                    null_model=null_model,
                    method=method    # <-- hinzugefügt
                )
                elapsed = time.time() - start_time
                print(f"⏱ {method} total runtime: {elapsed:.2f} Sekunden")

                if best_row is None:
                    print("⚠️ Keine optimalen Parameter gefunden.")
                    continue

                # 2. Beste Parameter extrahieren
                param_cols = ["sharpness_base","sharpness_growth","decay","max_freq","weight_exponent","freq_sharpness_exp"]
                best_params = {col: best_row[col] for col in param_cols}
                best_params.update({"norm_x":1.0,"output_max":1.0})

                print("Beste Parameter:", best_params)
                print("Optimierungswert:", best_row.to_dict())

                # 3. Alle Sequenzen im Ordner mit besten Parametern checken
                df_seq = analyze_folder_global_null(
                    folder_path=folder,
                    params=best_params,
                    num_sequences=num_sequences,
                    seed=seed,
                    method=method   # <-- hinzugefügt
                )
                df_seq["folder"] = folder
                df_seq["method"] = method
                df_seq["null_model"] = null_model
                df_seq["objective"] = objective
                all_seq_results.append(df_seq)

    # alles zusammenfassen
    if all_seq_results:
        final_df = pd.concat(all_seq_results, ignore_index=True)
    else:
        final_df = pd.DataFrame()
    return final_df

# ================================
# Beispielhafte Ausführung
# ================================

if __name__ == "__main__":
    folders = ["/Users/emilschmiedl/Desktop/Proximity-Funktion/data/Bat Carollia perspicillata (C_persp)"]
    
    # 1) Optimierung durchführen
    df_final = optimize_and_collect(folders, n_calls=80, num_sequences=200, seed=123, objective="pval")
    print("Finale Ergebnisse:")
    print(df_final.head())

    # 2) Benchmark anhand der realen IOI-Daten
    print("Benchmark der Proximity-Funktionen auf realen IOIs:")
    for folder in folders:
        for fname in os.listdir(folder):
            if not fname.endswith(".csv"):
                continue

            df = pd.read_csv(os.path.join(folder, fname))
            if "IOI" not in df.columns:
                continue

            iois = df["IOI"].dropna().values
            if len(iois) < 2:
                continue

            # Abstands-Matrix vorbereiten (2D für die Funktionen)
            X = iois.reshape(-1, 1)

            for name, func in [
                ("cosine", proximity_cosine),
                ("gaussian", proximity_gaussian),
            ]:
                start = time.time()
                sim_mat = func(X, sharpness_base=10.0, sharpness_growth=2.0, decay=0.1, max_freq=4,
                               weight_exponent=1.0, freq_sharpness_exp=1.0, norm_x=1.0, output_max=1.0)
                elapsed = time.time() - start
                print(f"{name:10s} ({fname}): {elapsed:.4f} Sekunden für {len(iois)} IOIs")


Starte Optimierung: Folder=/Users/emilschmiedl/Desktop/Proximity-Funktion/data/Bat Carollia perspicillata (C_persp), Methode=cosine, Nullmodell=expon, Objective=pval
[Bayesian Optimization] Iteration 1 / 80 (ETA: 0m 0s)
Global median IOI = 0.044
Uniform bounds = [0.019, 0.058]


/var/folders/3q/_wl3lf3179gcd3z2kdgr6g940000gn/T/ipykernel_14168/1935834374.py:42: RuntimeWarning: invalid value encountered in power
  sim = np.cos(np.minimum(x, max_freq) * np.pi / max_freq) ** weight_exponent


[Bayesian Optimization] Iteration 2 / 80 (ETA: 3m 0s)


/var/folders/3q/_wl3lf3179gcd3z2kdgr6g940000gn/T/ipykernel_14168/1935834374.py:42: RuntimeWarning: invalid value encountered in power
  sim = np.cos(np.minimum(x, max_freq) * np.pi / max_freq) ** weight_exponent


[Bayesian Optimization] Iteration 3 / 80 (ETA: 3m 2s)


...

[Bayesian Optimization] Iteration 80 / 80 (ETA: 0m 0s)


/var/folders/3q/_wl3lf3179gcd3z2kdgr6g940000gn/T/ipykernel_14168/1935834374.py:42: RuntimeWarning: invalid value encountered in power
  sim = np.cos(np.minimum(x, max_freq) * np.pi / max_freq) ** weight_exponent


⏱ cosine total runtime: 157.15 Sekunden
Beste Parameter: {'sharpness_base': np.float64(21.135294262741933), 'sharpness_growth': np.float64(2.898471485113051), 'decay': np.float64(0.5598030505215382), 'max_freq': np.float64(3.0), 'weight_exponent': np.float64(2.509268037350467), 'freq_sharpness_exp': np.float64(1.2966035735181645), 'norm_x': 1.0, 'output_max': 1.0}
Optimierungswert: {'norm_x': 1.0, 'output_max': 1.0, 'sharpness_base': 21.135294262741933, 'sharpness_growth': 2.898471485113051, 'decay': 0.5598030505215382, 'max_freq': 3.0, 'weight_exponent': 2.509268037350467, 'freq_sharpness_exp': 1.2966035735181645, 'agg_pval_empirical': 0.004975124378109453}


/var/folders/3q/_wl3lf3179gcd3z2kdgr6g940000gn/T/ipykernel_14168/1935834374.py:42: RuntimeWarning: invalid value encountered in power
  sim = np.cos(np.minimum(x, max_freq) * np.pi / max_freq) ** weight_exponent


Starte Optimierung: Folder=/Users/emilschmiedl/Desktop/Proximity-Funktion/data/Bat Carollia perspicillata (C_persp), Methode=gaussian, Nullmodell=expon, Objective=pval
[Bayesian Optimization] Iteration 1 / 80 (ETA: 0m 0s)
[Bayesian Optimization] Iteration 2 / 80 (ETA: 0m 55s)
[Bayesian Optimization] Iteration 3 / 80 (ETA: 1m 8s)
[Bayesian Optimization] Iteration 4 / 80 (ETA: 1m 22s)
[Bayesian Optimization] Iteration 5 / 80 (ETA: 1m 22s)
[Bayesian Optimization] Iteration 6 / 80 (ETA: 1m 23s)
[Bayesian Optimization] Iteration 7 / 80 (ETA: 1m 25s)
[Bayesian Optimization] Iteration 8 / 80 (ETA: 1m 25s)
[Bayesian Optimization] Iteration 9 / 80 (ETA: 1m 25s)
[Bayesian Optimization] Iteration 10 / 80 (ETA: 1m 25s)
[Bayesian Optimization] Iteration 11 / 80 (ETA: 1m 25s)
[Bayesian Optimization] Iteration 12 / 80 (ETA: 1m 24s)
[Bayesian Optimization] Iteration 13 / 80 (ETA: 1m 24s)
[Bayesian Optimization] Iteration 14 / 80 (ETA: 1m 24s)
[Bayesian Optimization] Iteration 15 / 80 (ETA: 1m 24s)
[Ba